# Raqamlar: modelni ko'zga ko'rinmas o'zgarish bilan aldash

Qo'lda yozilgan raqamlarni taniydigan model bor. Bu safar u «oq quti»: og'irliklari qo'lingizda, ichida nima
bo'layotganini to'liq ko'ra olasiz. Vazifangiz rasmning har bir pikselini juda oz, ko'z deyarli sezmaydigan
darajada o'zgartirib, modelni adashtirish. Odam rasmda hali ham «7» ni ko'radi, model esa «2» deydi.

Bunday rasmlar **adversarial example** deyiladi. Ular tasodifan paydo bo'lmaydi: hujumchi modelning o'zidan
«qaysi pikselni qayoqqa sursam, sen ko'proq adashasan» deb so'raydi. Bu savolning javobi **gradient** deb ataladi.

Bu darsda uch narsani o'rganasiz:

1. Gradient hujumchiga nimani aytib berishini.
2. Bir qadamli hujumni (FGSM) va u nega ba'zan yetmasligini.
3. Ko'p qadamli hujumni (PGD): kichik qadamlar bilan yurib, har qadamda yo'lni to'g'rilashni.

Python va `numpy` massivlari bilan biroz tanish bo'lsangiz yetarli. Hosila yoki gradientni hisoblashni bilish
shart emas: kerakli funksiya tayyor beriladi.

Bir gap: bu o'quv muhiti. Birovning tizimiga ruxsatsiz hujum qilish jinoyat hisoblanadi.

## 1. Labga ulanamiz

Platformadagi lab sahifasini oching. U yerda **Token** degan maydon bor: undagi satrni nusxalab, pastdagi
`TOKEN` ichiga qo'ying. Token faqat sizga tegishli va bir necha soatdan keyin eskiradi. Eskirib qolsa, lab
sahifasini yangilab, yangisini oling.

In [ ]:
BASE = "https://platform.hacknow.uz/ai-poligon"
TOKEN = ""   # lab sahifasidagi Token maydonidan nusxalab, qo'shtirnoq ichiga qo'ying

assert TOKEN, "Avval TOKEN ni to'ldiring: platformadagi lab sahifasidan nusxalang."

import urllib.request

import numpy as np

urllib.request.urlretrieve(BASE + "/client/hn_lab.py", "hn_lab.py")   # labning kichik mijozi
from hn_lab import HnLab

lab = HnLab(BASE, "ml-raqamlar-fgsm", TOKEN)
info = lab.task()
print(info["lab"]["title"])
print("Urinish:", info["attempt"], "| sarflangan so'rov:", info["queries_used"], "/", info["query_budget"])
if info["solved"]:
    print("Bu lab sizda allaqachon yechilgan. Flag:", info.get("flag"))
if info["query_budget"] - info["queries_used"] < 5:
    print("Diqqat: so'rovlar kam qoldi. Yetmay qolsa, lab.reset() bilan yangi urinish boshlang va notebook'ni boshidan yurgizing.")

## 2. Vazifa bilan tanishamiz

Sizga 20 ta rasm berilgan. Har bir rasm 8x8 piksel, ya'ni 64 ta son, har biri 0 (qora) bilan 1 (oq) orasida.
Model har rasm uchun 0 dan 9 gacha bo'lgan raqamlardan birini tanlaydi.

Ikkita fayl ochiq: `model.npz` ichida modelning og'irliklari, `images.npz` ichida rasmlar (`X`) va to'g'ri
javoblar (`y`) bor. Fayllarni yuklab olish so'rov sarflamaydi.

In [ ]:
vazifa = info["task"]
eps = vazifa["eps_linf"]

model = lab.npz("model.npz")
malumot = lab.npz("images.npz")
X0 = malumot["X"].astype(float)   # asl rasmlar: 20 qator, har qatorda 64 piksel
y = malumot["y"].astype(int)      # to'g'ri javoblar

print("Rasmlar:", X0.shape, "| to'g'ri javoblar:", y.tolist())
print("Har piksel ko'pi bilan shuncha o'zgarishi mumkin (eps):", eps)
print("Kamida shuncha rasmda model adashishi kerak:", vazifa["required_fooled"], "/", vazifa["n_images"])

**Cheklov nima?** Har bir piksel asl qiymatidan ko'pi bilan `eps` ga, ya'ni 0.12 ga farq qilishi mumkin.
Piksel 0 dan 1 gacha o'zgarishini eslasak, bu juda kichik siljish: rasm ko'zga deyarli o'sha-o'sha ko'rinadi.
Bundan tashqari, piksel 0 dan past yoki 1 dan baland bo'la olmaydi.

So'rovlar cheklangan: bitta urinishda 50 marta javob yubora olasiz. So'rovlar tugab qolsa, `lab.reset()` yangi
urinish boshlaydi, lekin unda rasmlar ham yangilanadi.

Keyingi katak birinchi rasmni chizadi. Agar kompyuteringizda `matplotlib` o'rnatilmagan bo'lsa, katak
shunchaki o'tkazib yuboriladi: dars rasmsiz ham davom etadi.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib o'rnatilmagan: rasm chizilmaydi, lekin dars davom etadi.")

if plt:
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(X0[0].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    plt.title("Birinchi rasm, to'g'ri javob: %d" % y[0])
    plt.axis("off")
    plt.show()

## 3. Model va gradient

Model oq quti bo'lgani uchun uning javobini o'zimiz hisoblay olamiz. Buning uchun serverga so'rov yuborish
shart emas. Pastda ikkita tayyor funksiya bor.

`bashorat` har rasm uchun model tanlagan raqamni qaytaradi.

`gradient` har rasmning har pikseli uchun bitta son qaytaradi. Bu son shunday o'qiladi: musbat bo'lsa, shu
pikselni oqartirsangiz model ko'proq adashadi; manfiy bo'lsa, qoraytirsangiz ko'proq adashadi. Bizga sonning
kattaligi emas, faqat ishorasi kerak bo'ladi: har pikselni qaysi tomonga surishni shu ishora aytadi.

In [ ]:
def bashorat(model, X):
    yashirin = np.maximum(X @ model["W1"] + model["b1"], 0.0)
    return (yashirin @ model["W2"] + model["b2"]).argmax(axis=1)


def gradient(model, X, y):
    """Har piksel uchun: uni oshirsak, modelning xatosi qanchaga o'sadi."""
    z1 = X @ model["W1"] + model["b1"]
    chiqish = np.maximum(z1, 0.0) @ model["W2"] + model["b2"]
    e = np.exp(chiqish - chiqish.max(axis=1, keepdims=True))
    ehtimol = e / e.sum(axis=1, keepdims=True)
    ehtimol[np.arange(len(y)), y] -= 1.0
    return ((ehtimol @ model["W2"].T) * (z1 > 0)) @ model["W1"].T


print("Asl rasmlarda model to'g'ri topgani:", int((bashorat(model, X0) == y).sum()), "/", len(y))

## 4. Birinchi urinish: bitta katta qadam (FGSM)

Eng oddiy hujum shunday: gradientni bir marta hisoblaymiz va har pikselni ishora ko'rsatgan tomonga ruxsat
etilgan eng katta masofaga, ya'ni `eps` ga suramiz. Keyin piksellar 0 bilan 1 orasida qolishi uchun chetga
chiqqanlarini qirqib tashlaymiz. Bu usul FGSM deb ataladi.

Model oq quti bo'lgani uchun nechta rasm aldanganini avval o'zimiz sanaymiz, keyin natijani serverga yuboramiz.

In [ ]:
X_fgsm = X0 + eps * np.sign(gradient(model, X0, y))
X_fgsm = np.clip(X_fgsm, 0.0, 1.0)

print("FGSM: aldangan rasmlar (o'zimiz sanadik):", int((bashorat(model, X_fgsm) != y).sum()), "/", len(y))

javob = lab.score({"images": X_fgsm.tolist()})
print(javob["output"])
print(javob["message"])
print("Sarflangan so'rov:", javob["queries_used"], "/", javob["query_budget"])

Bitta qadam yetmadi: ba'zi rasmlar bardosh berdi.

Nega? Gradient yo'lni faqat hozir turgan nuqtangizda to'g'ri ko'rsatadi. FGSM esa shu yo'nalishda birdaniga
katta sakraydi. Yo'l egri bo'lsa, sakrash mo'ljaldan o'tib ketadi. Tog'dan tumanda tushayotganingizni tasavvur
qiling: oyog'ingiz ostidagi qiyalikka qarab bitta juda uzun qadam tashlasangiz, boshqa tepalikka chiqib
qolishingiz mumkin. Mayda qadamlar bilan yurib, har qadamda qiyalikka qaytadan qarasangiz, pastga aniq tushasiz.

## 5. Topshiriq: ko'p qadamli hujum (PGD)

PGD aynan shuni qiladi: kichik qadamlar bilan yuradi va har qadamda gradientni qaytadan hisoblaydi. Har
qadamdan keyin ikki qoidani tiklash kerak, chunki qadamlar yig'ilib rasmni ruxsat etilgan chegaradan olib
chiqib ketishi mumkin:

1. Rasm asl rasmdan `eps` dan uzoqlashmasin. Buning uchun `np.clip(X, X0 - eps, X0 + eps)` ishlatiladi: u har
   pikselni asl qiymat atrofidagi ruxsat etilgan oraliqqa qaytaradi.
2. Piksellar 0 bilan 1 orasida qolsin: `np.clip(X, 0.0, 1.0)`.

Pastdagi katakda 40 marta takrorlanadigan sikl yozing. Har takrorda uch ish qilinadi: `X` ni gradient ishorasi
tomonga `qadam` ga surasiz (4-bo'limdagi birinchi satrga o'xshaydi, faqat `eps` o'rnida `qadam`, `X0` o'rnida
`X` turadi), keyin birinchi qoidani, undan keyin ikkinchi qoidani tiklaysiz.

In [ ]:
qadam = eps / 8   # bitta kichik qadamning uzunligi
X = X0.copy()     # asl rasmlardan boshlaymiz

# TOPSHIRIQ: siklni shu yerga yozing.

Keyingi katak natijani avval o'zi tekshiradi: cheklov buzilmaganmi va yetarlicha rasm aldanganmi. Hammasi
joyida bo'lsagina serverga yuboradi, shuning uchun so'rov bekorga ketmaydi.

In [ ]:
farq = float(np.abs(X - X0).max())
aldangan = int((bashorat(model, X) != y).sum())

if np.array_equal(X, X0):
    print("Hali hech narsa o'zgarmadi: Topshiriqni bajaring.")
elif farq > eps + 1e-9 or X.min() < 0.0 or X.max() > 1.0:
    print("Xato: cheklov buzilgan (eng katta farq %.3f, ruxsat %.2f). Ikki qoidani har qadamdan keyin tikladingizmi?" % (farq, eps))
elif aldangan < vazifa["required_fooled"]:
    print("Hali yetarli emas: %d ta rasm aldandi, kerak %d. Sikl 40 marta aylanyaptimi, gradient X dan hisoblanyaptimi?"
          % (aldangan, vazifa["required_fooled"]))
else:
    print("PGD: aldangan rasmlar (o'zimiz sanadik):", aldangan, "/", len(y), "| eng katta farq: %.3f" % farq)
    javob = lab.score({"images": X.tolist()})
    print(javob["output"])
    print(javob["message"])
    if javob["solved"]:
        print("Bo'ldi! Model yetarlicha rasmda adashdi.")
        print("Flag:", javob["flag"])

Flag chiqqan bo'lsa, uni platformadagi topshiriq sahifasiga yuboring. Flag faqat sizniki: boshqa odamning
flag'i sizda ishlamaydi.

Keyingi katak asl va o'zgartirilgan rasmni yonma-yon chizadi (`matplotlib` bo'lsa). Farqni ko'z bilan topishga
urinib ko'ring.

In [ ]:
if plt and not np.array_equal(X, X0):
    k = int(np.argmax(bashorat(model, X) != y))   # birinchi aldangan rasm
    plt.figure(figsize=(6, 2.6))
    for joy, (rasm, sarlavha) in enumerate(((X0[k], "asl: model %d dedi" % bashorat(model, X0[k:k + 1])[0]),
                                            (X[k], "o'zgargan: model %d dedi" % bashorat(model, X[k:k + 1])[0]))):
        plt.subplot(1, 2, joy + 1)
        plt.imshow(rasm.reshape(8, 8), cmap="gray", vmin=0, vmax=1)
        plt.title(sarlavha)
        plt.axis("off")
    plt.show()

## 6. Nima bo'ldi?

Har pikselni ko'pi bilan 0.12 ga o'zgartirdingiz. Odam uchun rasm o'sha-o'sha, model esa deyarli hamma rasmda
boshqa raqamni aytdi. Sababi shundaki, model rasmga odamdek qaramaydi: u 64 ta sonning yig'indilariga tayanadi
va mayda, lekin aniq yo'naltirilgan o'zgarishlar bu yig'indilarni keskin siljitadi.

Bir qadamli FGSM tez, lekin qo'pol. PGD sekinroq, lekin ancha kuchli: bugungi kunda modelning bardoshliligini
tekshirishda aynan u asosiy o'lchov hisoblanadi.

Himoyachi nima qilishi mumkin? Eng ishonchli usul **adversarial training**: model o'qitilayotganda unga shunday
o'zgartirilgan rasmlar ham ko'rsatiladi va to'g'ri javob o'rgatiladi. Bu aniqlikni biroz pasaytiradi va
o'qitishni qimmatlashtiradi, lekin bardoshlilikni chindan oshiradi. Og'irliklarni yashirish esa yetarli himoya
emas: hujumchi o'ziga o'xshash model o'qitib, unda topgan rasmlarini sizning modelingizga yuborishi mumkin va
ular ko'pincha u yerda ham ishlaydi.

## Qo'shimcha (xohlasangiz)

1. Qadamlar sonini 5 ga tushirib ko'ring. Nechta rasm aldanadi?
2. `eps` ni kichraytirib ko'ring (masalan 0.06). Hujum qachon ishlamay qoladi?
3. Hozir model «istalgan noto'g'ri raqam»ni aytsa bo'ldi. Uni aynan siz xohlagan raqamni aytishga majburlasa
   bo'ladimi? Buning uchun gradient funksiyasiga qaysi javobni berish kerak va ishorani qanday o'zgartirish kerak?